## Two-Column Click Modeling

Current workflow:
1. Preprocessing
2. Position-based / probabilistic models
3. Neural extension
4. Logistic regression comparison
5. Interpretation of layout effects beyond ranking position

In [4]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt

ds1 = load_dataset("THUIR/Qilin", "recommendation_train")
ds2 = load_dataset("THUIR/Qilin", "recommendation_test")
ds3 = load_dataset("THUIR/Qilin", "notes")

df_train = ds1["train"].to_pandas()
df_test = ds2["train"].to_pandas()
# df_notes = ds3["train"].to_pandas()


print(df_train.shape)
print(df_train.columns)

print(df_test.shape)
print(df_test.columns)

# print(df_notes.shape)
# print(df_notes.columns)

notes/train-00000-of-00005.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

notes/train-00001-of-00005.parquet:   0%|          | 0.00/205M [00:00<?, ?B/s]

notes/train-00002-of-00005.parquet:   0%|          | 0.00/192M [00:00<?, ?B/s]

notes/train-00003-of-00005.parquet:   0%|          | 0.00/191M [00:00<?, ?B/s]

notes/train-00004-of-00005.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1983938 [00:00<?, ? examples/s]

(83437, 6)
Index(['recent_clicked_note_idxs', 'request_idx', 'session_idx', 'user_idx',
       'query', 'rec_result_details_with_idx'],
      dtype='str')
(11115, 7)
Index(['recent_clicked_note_idxs', 'request_idx', 'session_idx', 'user_idx',
       'rec_results', 'query', 'rec_result_details_with_idx'],
      dtype='str')
(1983938, 44)
Index(['note_title', 'note_content', 'note_type', 'video_duration',
       'video_height', 'video_width', 'image_num', 'content_length',
       'commercial_flag', 'taxonomy1_id', 'taxonomy2_id', 'taxonomy3_id',
       'imp_num', 'imp_rec_num', 'imp_search_num', 'click_num',
       'click_rec_num', 'click_search_num', 'like_num', 'collect_num',
       'comment_num', 'share_num', 'screenshot_num', 'hide_num',
       'rec_like_num', 'rec_collect_num', 'rec_comment_num', 'rec_share_num',
       'rec_follow_num', 'search_like_num', 'search_collect_num',
       'search_comment_num', 'search_share_num', 'search_follow_num',
       'accum_like_num', 'accum_coll

In [6]:
def flatten_recommendation_df(df):
    result_df = (
        df[["session_idx", "request_idx", "rec_result_details_with_idx"]]
        .explode("rec_result_details_with_idx")
    )

    result_df = result_df[result_df["rec_result_details_with_idx"].notna()].copy()

    item_details = pd.json_normalize(result_df["rec_result_details_with_idx"])

    flat_df = pd.concat(
        [
            result_df[["session_idx", "request_idx"]].reset_index(drop=True),
            item_details.reset_index(drop=True)
        ],
        axis=1
    )

    return flat_df

In [ ]:
train_flat = flatten_recommendation_df(df_train)
test_flat = flatten_recommendation_df(df_test)


Index(['session_idx', 'request_idx', 'click', 'collect', 'comment', 'like',
       'note_idx', 'page_time', 'position', 'request_timestamp', 'share'],
      dtype='str')

In [16]:
train_flat["row"] = (train_flat["position"] - 1) // 2
train_flat["column"] = (train_flat["position"] - 1) % 2

test_flat["row"] = (test_flat["position"] - 1) // 2
test_flat["column"] = (test_flat["position"] - 1) % 2

In [17]:
train_flat.head()
train_flat.columns
test_flat.head()
test_flat.columns

Index(['session_idx', 'request_idx', 'click', 'collect', 'comment', 'like',
       'note_idx', 'page_time', 'position', 'request_timestamp', 'share',
       'row', 'column'],
      dtype='str')

In [ ]:
# train_flat.isna().sum()
# test_flat.isna().sum()
# train_flat.isna().mean().sort_values(ascending=False)
# test_flat.isna().mean().sort_values(ascending=False)
# train_flat["click"].value_counts()


count    182565.000000
mean          0.058211
std           0.055489
min           0.000000
25%           0.033243
50%           0.049308
75%           0.074433
max           0.467751
Name: p_click, dtype: float64

## 1D PBM Model

In [28]:
gamma = train_flat.groupby("position")["click"].mean()

In [29]:
alpha = train_flat.groupby("note_idx")["click"].mean()
alpha_global = train_flat["click"].mean()

In [33]:
test_flat["gamma"] = test_flat["position"].map(gamma)
test_flat["alpha"] = test_flat["note_idx"].map(alpha)

test_flat["alpha"] = test_flat["alpha"].fillna(alpha_global)

test_flat["p_click"] = test_flat["gamma"] * test_flat["alpha"]

test_flat["p_click"].describe()

count    182565.000000
mean          0.058211
std           0.055489
min           0.000000
25%           0.033243
50%           0.049308
75%           0.074433
max           0.467751
Name: p_click, dtype: float64

In [31]:
import numpy as np

def log_likelihood(y, p):
    eps = 1e-10
    p = p.clip(eps, 1 - eps)
    return (y * np.log(p) + (1 - y) * np.log(1 - p)).mean()

score_1d = log_likelihood(test_flat["click"], test_flat["p_click"])

print("1D PBM Log-Likelihood:", score_1d)

1D PBM Log-Likelihood: -1.478721781160754


## 2D PBM Model

### 2D Separate

In [34]:
gamma_row = train_flat.groupby("row")["click"].mean()
gamma_col = train_flat.groupby("column")["click"].mean()

gamma_row = gamma_row / gamma_row.mean()
gamma_col = gamma_col / gamma_col.mean()

In [35]:
test_flat["gamma_row"] = test_flat["row"].map(gamma_row)
test_flat["gamma_col"] = test_flat["column"].map(gamma_col)

test_flat["gamma_sep"] = test_flat["gamma_row"] * test_flat["gamma_col"]

In [36]:
test_flat["p_sep"] = test_flat["gamma_sep"] * test_flat["alpha"]

In [37]:
score_sep = log_likelihood(test_flat["click"], test_flat["p_sep"])
print("2D Separate Log-Likelihood:", score_sep)

2D Separate Log-Likelihood: -2.7921414279027177


### 2D Full (row x column interaction)

In [55]:
gamma_2d = train_flat.groupby(["row", "column"])["click"].mean()


In [56]:
test_flat["gamma_2d"] = test_flat.set_index(["row", "column"]).index.map(gamma_2d)

In [57]:
test_flat["gamma_2d"] = test_flat["gamma_2d"].fillna(train_flat["click"].mean())

In [58]:
test_flat["p_2d"] = test_flat["gamma_2d"] * test_flat["alpha"]

In [59]:
score_2d = log_likelihood(test_flat["click"], test_flat["p_2d"])
print("2D Full Log-Likelihood:", score_2d)

2D Full Log-Likelihood: -1.478705914574431


In [60]:
print("1D:", score_1d)
print("2D Separate:", score_sep)
print("2D Full:", score_2d)

1D: -1.478721781160754
2D Separate: -2.7921414279027177
2D Full: -1.478705914574431


### 2D Full Improved

In [63]:
gamma_row = train_flat.groupby("row")["click"].mean()

gamma_2d = (
    train_flat.groupby(["row","column"])["click"].mean()
    / train_flat.groupby("row")["click"].mean()
)

test_flat["gamma_row"] = test_flat["row"].map(gamma_row)

test_flat["col_pref"] = test_flat.set_index(["row","column"]).index.map(gamma_2d)
test_flat["col_pref"] = test_flat["col_pref"].fillna(1.0)

test_flat["p_2d_new"] = test_flat["gamma_row"] * test_flat["col_pref"] * test_flat["alpha"]

score_2d_new = log_likelihood(test_flat["click"], test_flat["p_2d_new"])
print("2D Full improved:", score_2d_new)

2D Full improved: -1.4787055819376624


In [64]:
print("1D:", score_1d)
print("2D Separate:", score_sep)
print("2D Full:", score_2d)
print("2D Full improved:", score_2d_new)

1D: -1.478721781160754
2D Separate: -2.7921414279027177
2D Full: -1.478705914574431
2D Full improved: -1.4787055819376624


In [66]:
n = len(test_flat)

print("Total improvement:", (score_2d_new - score_1d) * n)

Total improvement: 2.9574435621632738


Although the average log-likelihood improvement per impression is small, the improvement accumulates across the full test set. The improved 2D model gains approximately 2.96 total log-likelihood points over the 1D baseline, indicating that row-column structure provides additional predictive information beyond linear position alone.

## Current Progress

### 1D Baseline
- Implemented a 1D position-based baseline using ranking position only
- Position already captures most of the predictive signal

---

### 2D Separate Model
Assumption:
- Row and column effects are independent

Findings:
- Performed worse than the 1D baseline
- Suggests that row and column cannot simply be modeled independently

Interpretation:
- Column preference likely depends on row context

---

### 2D Full Model
Approach:
- Treats each `(row, column)` combination as a separate position
- Essentially a lookup-table formulation

Findings:
- Performs almost the same as the 1D baseline

Interpretation:
- Most information is already captured by ranking position
- Limited interpretability / behavioral insight

---

### 2D Improved / Interaction Model
Approach:
- Explicitly models row-column interaction structure
- Assumes users scan vertically through rows and then choose left/right within a row

Findings:
- Small improvement over the 1D baseline
- Best-performing probabilistic/logistic formulation so far

Interpretation:
- User attention is largely determined by ranking position
- However, row-column interactions still contain limited additional predictive information

---

## Logistic Regression Results

| Model | Log Loss | AUC |
|---|---|---|
| 1D Logistic | 0.538477 | 0.612184 |
| 2D Separate Logistic | 0.539451 | 0.607346 |
| 2D Interaction Logistic | 0.538460 | 0.612202 |

Main observation:
- Naive independent 2D assumptions do not improve performance
- Explicit interaction modeling gives a small but consistent improvement

---

## Neural Extension
- Implemented an initial neural click model using:
  - row
  - column
  - item embeddings (`note_idx`)

Current results:
- Log Loss: ~0.635
- AUC: ~0.587

Current interpretation:
- Neural model fits probabilities substantially better than the earlier PBM-style implementation
- Still investigating how much improvement comes from layout information versus item popularity / embeddings
- Possible next step:
  - compare item-only neural model vs item + row/column model

## Neural Extension

- Implemented an initial neural click prediction model using:
  - row
  - column
  - item embeddings (`note_idx`)

- Goal:
  - investigate whether a more flexible model can capture additional layout structure and improve predictive performance beyond probabilistic position-based models

- Initial findings:
  - substantially improved likelihood compared to the earlier PBM-style implementation
  - however, predictive power remains moderate overall
  - current results suggest that part of the improvement may come from item embeddings/popularity effects rather than layout information alone

- Possible next step:
  - compare:
    - item-only neural model
    - item + row/column neural model

In [67]:
import torch
import torch.nn as nn

class ClickModel(nn.Module):
    def __init__(self, num_items, embedding_dim=16):
        super().__init__()
        
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        
        self.fc = nn.Sequential(
            nn.Linear(embedding_dim + 2, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    
    def forward(self, row, column, item):
        emb = self.item_emb(item)
        x = torch.cat([emb, row.unsqueeze(1), column.unsqueeze(1)], dim=1)
        return self.fc(x).squeeze()

In [68]:

# map note_idx to integers
note_to_id = {n:i for i,n in enumerate(train_flat["note_idx"].unique())}

train_flat["item_id"] = train_flat["note_idx"].map(note_to_id)
test_flat["item_id"] = test_flat["note_idx"].map(note_to_id).fillna(0)

In [69]:
model = ClickModel(num_items=len(note_to_id))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

for epoch in range(5):  # keep small
    model.train()
    
    row = torch.tensor(train_flat["row"].values, dtype=torch.float32)
    col = torch.tensor(train_flat["column"].values, dtype=torch.float32)
    item = torch.tensor(train_flat["item_id"].values, dtype=torch.long)
    y = torch.tensor(train_flat["click"].values, dtype=torch.float32)
    
    pred = model(row, col, item)
    loss = loss_fn(pred, y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch}: {loss.item()}")

Epoch 0: 0.6726200580596924
Epoch 1: 0.6659461855888367
Epoch 2: 0.6595759987831116
Epoch 3: 0.6534934043884277
Epoch 4: 0.6476607322692871


In [70]:
model.eval()

row = torch.tensor(test_flat["row"].values, dtype=torch.float32)
col = torch.tensor(test_flat["column"].values, dtype=torch.float32)
item = torch.tensor(test_flat["item_id"].values, dtype=torch.long)

with torch.no_grad():
    p_nn = model(row, col, item).numpy()

score_nn = log_likelihood(test_flat["click"], p_nn)

print("Neural model LL:", score_nn)

Neural model LL: -0.6348480665054642


In [71]:
test_flat["p_nn"] = p_nn
test_flat["p_nn"].describe()

from sklearn.metrics import roc_auc_score, log_loss

print("NN Log loss:", log_loss(test_flat["click"], test_flat["p_nn"]))
print("NN AUC:", roc_auc_score(test_flat["click"], test_flat["p_nn"]))

NN Log loss: 0.6348480647762844
NN AUC: 0.5867540098151754


## Logistic Regression Experiment

Does adding row/column improve prediction vs position?

### 1D log reg

In [73]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [74]:
X_train_1d = train_flat[["position"]]
X_test_1d = test_flat[["position"]]

y_train = train_flat["click"]
y_test = test_flat["click"]

In [75]:
model_1d = Pipeline([
    ("encoder", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["position"])
    ])),
    ("clf", LogisticRegression(max_iter=1000))
])

In [76]:
model_1d.fit(X_train_1d, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('encoder', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrice

In [77]:
p_1d = model_1d.predict_proba(X_test_1d)[:, 1]

In [78]:
print("1D Log Loss:", log_loss(y_test, p_1d))
print("1D AUC:", roc_auc_score(y_test, p_1d))

1D Log Loss: 0.5384774389182732
1D AUC: 0.6121839029647257


### 2D Seperate log reg

In [86]:
X_train_2d = train_flat[["row", "column"]]
X_test_2d = test_flat[["row", "column"]]

In [87]:
model_2d = Pipeline([
    ("encoder", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["row", "column"])
    ])),
    ("clf", LogisticRegression(max_iter=1000))
])

In [88]:
model_2d.fit(X_train_2d, y_train)

p_2d = model_2d.predict_proba(X_test_2d)[:, 1]

In [89]:
print("2D Log Loss:", log_loss(y_test, p_2d))
print("2D AUC:", roc_auc_score(y_test, p_2d))

2D Log Loss: 0.5394510844625499
2D AUC: 0.6073463249059607


### 2D Full log reg 

In [95]:
train_flat["row_col"] = (
    train_flat["row"].astype(str)
    + "_"
    + train_flat["column"].astype(str)
)

test_flat["row_col"] = (
    test_flat["row"].astype(str)
    + "_"
    + test_flat["column"].astype(str)
)

In [102]:
X_train_inter = train_flat[["row", "column", "row_col"]]
X_test_inter = test_flat[["row", "column", "row_col"]]

In [103]:
model_inter = Pipeline([
    ("encoder", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         ["row", "column", "row_col"])
    ])),
    ("clf", LogisticRegression(max_iter=1000))
])

In [104]:
model_inter.fit(X_train_inter, y_train)

p_inter = model_inter.predict_proba(X_test_inter)[:, 1]

In [105]:
print("2D Interaction Log Loss:", log_loss(y_test, p_inter))
print("2D Interaction AUC:", roc_auc_score(y_test, p_inter))

2D Interaction Log Loss: 0.5384603208888372
2D Interaction AUC: 0.6122018917447655


In [106]:
print("\n========== MODEL COMPARISON ==========")

print("\n1D Logistic Model")
print("Log Loss:", log_loss(y_test, p_1d))
print("AUC:", roc_auc_score(y_test, p_1d))

print("\n2D Separate Logistic Model")
print("Log Loss:", log_loss(y_test, p_2d))
print("AUC:", roc_auc_score(y_test, p_2d))

print("\n2D Interaction Logistic Model")
print("Log Loss:", log_loss(y_test, p_inter))
print("AUC:", roc_auc_score(y_test, p_inter))


========== MODEL COMPARISON ==========

1D Logistic Model
Log Loss: 0.5384774389182732
AUC: 0.6121839029647257

2D Separate Logistic Model
Log Loss: 0.5394510844625499
AUC: 0.6073463249059607

2D Interaction Logistic Model
Log Loss: 0.5384603208888372
AUC: 0.6122018917447655


## Results

### Position-Based Models

Position Based Model log-likelihood:  

1D: -1.478721781160754  
2D Separate: -2.7921414279027177  
2D Full: -1.478705914574431  
2D Full Improved: -1.4787055819376624  

---

### Logistic Regression Models

1D Logistic Model  
- Log Loss: 0.5384774389182732  
- AUC: 0.6121839029647257  

2D Separate Logistic Model  
- Log Loss: 0.5394510844625499  
- AUC: 0.6073463249059607  

2D Interaction Logistic Model  
- Log Loss: 0.5384603208888372  
- AUC: 0.6122018917447655  

---

### Neural Extension

Neural Model (row + column + item embeddings)  
- Log-Likelihood: -0.6348480665054642  
- Log Loss: 0.6348480647762844  
- AUC: 0.5867540098151754  

---

### Main Observations

- Simple independent 2D assumptions (separate row and column effects) do not improve performance over the 1D baseline.

- Explicitly modeling row-column interactions yields a small but consistent improvement over the 1D model.

- This suggests that user attention is largely determined by ranking position, while row-column layout adds only limited additional predictive value.

- At the same time, the interaction model indicates that the exact row-column combination still contains some additional predictive information beyond position alone.

- The neural extension substantially improves likelihood compared to the earlier PBM-style implementation, although predictive power remains moderate overall.